In [ ]:
#TSA for error annotation
import os
import json
import asyncio
from openai import AsyncOpenAI

# ----------------------------
# CONFIGURATION
# ----------------------------
TEXT_FOLDER = "Texts"
OUTPUT_FOLDER = "Annotations"
MODEL = "gpt-5.2-2025-12-11"

client = AsyncOpenAI()

# ----------------------------
# FEW-SHOT EXAMPLES
# ----------------------------
few_shots = {
    "ende": {
            "source_lang": "English",
            "source_seg": "I do apologise about this, we must gain permission from the account holder to discuss an order with another person, I apologise if this was done previously, however, I would not be able to discuss this with yourself without the account holders permission.",
            "target_lang": "German",
            "target_seg": "Ich entschuldige mich dafür, wir müssen die Erlaubnis einholen, um eine Bestellung mit einer anderen Person zu besprechen. Ich entschuldige mich, falls dies zuvor geschehen wäre, aber ohne die Erlaubnis des Kontoinhabers wäre ich nicht in der Lage, dies mit dir involvement.",
            "answer": '''{
"annotated_translation": "Ich entschuldige mich dafür, wir müssen die Erlaubnis<v0></v0> einholen, um eine Bestellung mit einer anderen Person zu besprechen. Ich entschuldige mich, falls dies zuvor geschehen <v1>wäre</v1>, aber ohne die Erlaubnis des Kontoinhabers wäre ich nicht in der Lage, dies mit <v2>dir</v2> <v3>involvement</v3>.",
"errors": [
    {"severity": "Major", "category": "accuracy/omission"},
    {"severity": "Minor", "category": "fluency/grammar"},
    {"severity": "Minor", "category": "fluency/register"},
    {"severity": "Major", "category": "accuracy/mistranslation"}
]
}'''
        },
    "encs": {
            "source_lang": "English",
            "source_seg": "Talks have resumed in Vienna to try to revive the nuclear pact, with both sides trying to gauge the prospects of success after the latest exchanges in the stop-start negotiations.",
            "target_lang": "Czech",
            "target_seg": "Ve Vídni se ve Vídni obnovily rozhovory o oživení jaderného paktu, přičemž obě partaje se snaží posoudit vyhlídky na úspěch po posledních výměnách v jednáních.",
            "answer": '''{
"annotated_translation": "Ve Vídni se <v0>ve Vídni</v0> obnovily rozhovory o oživení jaderného paktu, přičemž obě <v1>partaje</v1> se snaží posoudit vyhlídky na úspěch po posledních výměnách v<v2></v2> jednáních.",
"errors": [
    {"severity": "Major", "category": "accuracy/addition"},
    {"severity": "Minor", "category": "terminology/inappropriate for context"},
    {"severity": "Major", "category": "accuracy/omission"}
]
}'''
        },
    "zhen": {
            "source_lang": "Chinese",
            "source_seg": "大众点评乌鲁木齐家居卖场频道为您提供高铁居然之家地址，电话，营业时间等最新商户信息，找装修公司，就上大众点评",
            "target_lang": "English",
            "target_seg": "Urumqi Home Furnishing Store Channel provides you with the latest business information such as the address, telephone number, business hours, etc., of high-speed rail, and find a decoration company, and go to the reviews.",
            "answer": '''{
"annotated_translation": "Urumqi Home Furnishing Store Channel provides you with the latest business information such as the address, telephone number, business hours, <v0>etc.,</v0> <v1>of high-speed rail,</v1> and find a decoration company, and <v2>go to the reviews</v2>.",
"errors": [
    {"severity": "Minor", "category": "style/awkward"},
    {"severity": "Major", "category": "accuracy/addition"},
    {"severity": "Major", "category": "accuracy/mistranslation"}
]
}'''
        },
    "enes": {
            "source_lang": "English",
            "source_seg": "According to the terms outlined in the agreement, the supplier shall deliver all components no later than thirty days after receiving the initial purchase order, and any delays must be communicated in writing at least five business days in advance.",
            "target_lang": "Spanish",
            "target_seg": "De acuerdo con los términos establecidos en el acuerdo, el proveedor deberá entregar todos los componentes a más tardar treinta días después de recibir la orden de compra inicial, y cualquier retraso deberá comunicarse por escrito con al menos cinco días hábiles de antelación.",
            "answer": '''{
"annotated_translation": "De acuerdo con los términos establecidos en el acuerdo, el proveedor deberá entregar todos los componentes a más tardar treinta días después de recibir la orden de compra inicial, y cualquier retraso deberá comunicarse por escrito con al menos cinco días hábiles de antelación.",
"errors": [
]
}'''
        },
}

# ----------------------------
# PROMPT TEMPLATE
# ----------------------------

def escape_braces(text: str) -> str:
    return text.replace("{", "{{").replace("}", "}}")

FEW_SHOT_BLOCK = "\n\n### FEW-SHOT EXAMPLES:\n"
for ex in few_shots.values():
    FEW_SHOT_BLOCK += (
        f"\n\n# Source ({ex['source_lang']}):\n"
        f"{escape_braces(ex['source_seg'])}\n\n"
        f"# Translation ({ex['target_lang']}):\n"
        f"{escape_braces(ex['target_seg'])}\n\n"
        f"# Expected Output:\n{escape_braces(ex['answer'])}\n"
    )

PROMPT_TEMPLATE = """You are a careful and balanced annotator for machine translation quality.
Your task is to identify translation errors with appropriate confidence.

## EVALUATION GUIDELINES:
- Be thorough but precise
- Only mark errors when confident
- Focus on objective errors, not preferences
- Mark minimal error spans with <v0>, <v1>, ...
- Do NOT treat cross-sentence shifts as errors.
- If a phrase is moved to the previous/next sentence but translated correctly, do NOT label it as omission or addition.
- Only mark an omission/addition if the content is missing entirely or appears incorrectly in the translation as a whole, not just in this segment.
- Use the provided previous/next sentences to verify whether the content is simply moved rather than missing.

## Error Categories:
- Accuracy (addition, omission, mistranslation, overtranslation, undertranslation untranslated)
- Fluency (grammar, punctuation, spelling)
- Style (awkward, inconsistent, register, unidiomatic, audience appropriateness)
- Terminology
- Other

## Severity:
- Major
- Minor

## CRITICAL RULES:
- Tags must be sequential: <v0>, <v1>, <v2> ...
- No explanations, comments, or extra text
- If omission: insert empty tags <vN></vN>

""" + FEW_SHOT_BLOCK + """

### CONTEXT (NOT FOR EVALUATION):
Previous source: {prev_source}
Previous translation: {prev_translation}

Next source: {next_source}
Next translation: {next_translation}

### SOURCE TEXT (CURRENT SENTENCE):
{source}

### TRANSLATION (CURRENT SENTENCE):
{translation}
"""

# ----------------------------
# Model call
# ----------------------------
async def annotate_line(source, translation, prev_source, prev_translation, next_source, next_translation):
    prompt = PROMPT_TEMPLATE.format(
        source=source,
        translation=translation,
        prev_source=prev_source,
        prev_translation=prev_translation,
        next_source=next_source,
        next_translation=next_translation
    )

    response = await client.responses.create(
        model=MODEL,
        reasoning={"effort": "high"},
        input=prompt
    )

    return response.output_text


# ----------------------------
# Parse filename
# ----------------------------
def parse_filename(filename):
    name, _ = os.path.splitext(filename)
    try:
        st_tt, genre, modality = name.split("_")
        st, tt = st_tt.split("-")
        return st, tt, genre, modality
    except ValueError:
        return None


# ----------------------------
# Index files
# ----------------------------
def index_texts():
    index = {}
    for fname in os.listdir(TEXT_FOLDER):
        parsed = parse_filename(fname)
        if not parsed:
            continue

        st, tt, genre, mod = parsed
        key = (st, tt, genre)

        if key not in index:
            index[key] = {}

        index[key][mod] = os.path.join(TEXT_FOLDER, fname)

    return index


# ----------------------------
# MAIN WORKFLOW — SENTENCE-BY-SENTENCE
# ----------------------------
async def main():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    text_index = index_texts()

    for (st, tt, genre), files in text_index.items():
        if "ST" not in files:
            continue

        with open(files["ST"], "r", encoding="utf-8") as f:
            source_lines = f.read().splitlines()

        for modality in ["HT", "MT", "PE"]:
            if modality not in files:
                continue

            with open(files[modality], "r", encoding="utf-8") as f:
                trans_lines = f.read().splitlines()

            n = min(len(source_lines), len(trans_lines))

            print(f"Annotating {st}-{tt} {genre} {modality} ({n} lines)...")

            results = []

            for i in range(n):
                src = source_lines[i].strip()
                tgt = trans_lines[i].strip()

                if not src and not tgt:
                    results.append({
                        "line": i+1,
                        "source": src,
                        "translation": tgt,
                        "model_output_json": {}
                    })
                    continue

                prev_src = source_lines[i-1].strip() if i > 0 else ""
                next_src = source_lines[i+1].strip() if i < n-1 else ""
                prev_tgt = trans_lines[i-1].strip() if i > 0 else ""
                next_tgt = trans_lines[i+1].strip() if i < n-1 else ""

                annotation = await annotate_line(
                    src, tgt,
                    prev_src, prev_tgt,
                    next_src, next_tgt
                )

                # Try parse JSON
                try:
                    annotation_json = json.loads(annotation)
                except:
                    annotation_json = {"RAW_MODEL_OUTPUT": annotation}

                results.append({
                    "line": i+1,
                    "source": src,
                    "translation": tgt,
                    "model_output_json": annotation_json
                })

            out_file = f"{st}-{tt}_{genre}_{modality}_ANNOTATED.json"
            out_path = os.path.join(OUTPUT_FOLDER, out_file)

            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)

            print(f" → Saved {out_path}")

    print("Done!")


if __name__ == "__main__":
    asyncio.run(main())



In [ ]:
import asyncio

# Detect running loop
try:
    loop = asyncio.get_running_loop()
    print("Using running loop")
    # Await the main coroutine in the existing loop
    task = loop.create_task(main())
    await task  # <-- make sure to await it in Jupyter / IPython
except RuntimeError:
    print("No running loop, creating new one")
    asyncio.run(main())


In [ ]:
#Creativity annotation
import os
import csv
import json
import asyncio
from openai import AsyncOpenAI

# Initialize async OpenAI client
client = AsyncOpenAI()

SYSTEM_PROMPT = """
You are a careful and balanced annotator for creativity in translation.
Your task is to categorise potential creative segments (UCP) as a creative shift (CS), Reproduction (R), omission (O), or error (E)

## EVALUATION GUIDELINES:
- Be thorough but precise
- Consider whether there is a direct or coined translation available
- Do NOT treat cross-sentence shifts as errors.
- Only mark an omission/addition if the content is missing entirely or appears incorrectly in the translation as a whole, not just in this segment.

## Creativity Categories:
- Creative shift (CS) (All translations that deviate from the source with a different idea or image are considered creative shifts. These are the creative shifts (“non-literal”).) includes:
-- CSA (Abstraction): refers to those cases in which translators use more vague, general or abstract in the translation
-- CSM (Modification): refers to shifts that are at the same level of abstraction (e.g. express a source text metaphor with a different metaphor without the image becoming more abstract or concrete). In other words, the translation is modified for the target culture.
-- CSC (Concretisation): refers to instances when the translaton evokes a more explicit, more detailed and more precise idea or image than the source text
- Reproduction (R): translations that reproduce the source text with the same idea or image, even if they are acceptable, are not considered a creative shift in the translation, but a reproduction.
- Omission (O): If a term or expression in the source text is omitted in the translation this will be marked as omission. An omission could correspond to a) creative solution (for example, that text was omitted because it does not make sense in the translation) or b) a shortcut solution (for example, that text is omitted because it is rather cumbersome to render).
- Error (E): If the translation is not acceptable (contains too many errors).
"""

INPUT_FOLDER = "Texts"
OUTPUT_FOLDER = "Texts_annotated"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


async def annotate_line(source, translation, ucp, ucp_trans):
    """
    Send one UCP pair to GPT-5.2 and get back the annotation as JSON.
    """

    user_message = (
        f"Source sentence: {source}\n"
        f"Translation: {translation}\n"
        f"UCP: {ucp}\n"
        f"UCP translation: {ucp_trans}\n\n"
        "Return ONLY a JSON object with keys: 'ucp', 'ucp_translation', 'annotation'."
    )

    response = await client.responses.create(
        model="gpt-5.2-2025-12-11",
        reasoning={"effort": "high"},
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message}
        ]
    )

    return json.loads(response.output_text)


async def process_csv(file_path):
    """
    Read a CSV, annotate each row asynchronously, return a list of results.
    """

    annotations = []

    # Collect tasks to run concurrently
    tasks = []

    with open(file_path, encoding="utf-8") as f:
        reader = csv.reader(f, delimiter=";")
        next(reader)  # skip headers

        for row in reader:
            if len(row) < 4:
                continue

            source, translation, ucp, ucp_trans = row[:4]

            tasks.append(
                annotate_line(source, translation, ucp, ucp_trans)
            )

    # Run all annotation tasks concurrently
    if tasks:
        annotations = await asyncio.gather(*tasks)

    return annotations


async def main():
    for filename in os.listdir(INPUT_FOLDER):
        if not filename.endswith(".csv"):
            continue

        input_path = os.path.join(INPUT_FOLDER, filename)
        output_path = os.path.join(
            OUTPUT_FOLDER, filename.replace(".csv", ".json")
        )

        print(f"Processing {filename}...")

        annotations = await process_csv(input_path)

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(annotations, f, ensure_ascii=False, indent=2)

        print(f"Saved → {output_path}")


if __name__ == "__main__":
    asyncio.run(main())


In [ ]:
import asyncio

# Detect running loop
try:
    loop = asyncio.get_running_loop()
    print("Using running loop")
    # Await the main coroutine in the existing loop
    task = loop.create_task(main())
    await task  # <-- make sure to await it in Jupyter / IPython
except RuntimeError:
    print("No running loop, creating new one")
    asyncio.run(main())